In [37]:
import torchvision
from torchvision.models.detection.mask_rcnn import MaskRCNN_ResNet50_FPN_Weights, MaskRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.tensorboard import SummaryWriter

In [38]:
def get_model(num_classes: int = 2) -> MaskRCNN:
    """
    Create a Mask R-CNN model for instance segmentation.

    :param num_classes: Number of output classes including background (default 2 → background + ship).
    :returns: Mask R-CNN model with a ResNet-50 FPN backbone, customized for the specified number of classes.
    """
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = \
        torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden = 256
    model.roi_heads.mask_predictor = \
        torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
            in_features_mask, hidden, num_classes
        )

    return model

In [39]:
import torch
import torch.optim as optim

In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

model = get_model()
model.to(device)

optimizer = optim.SGD(
    model.parameters(),
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

writer = SummaryWriter(log_dir="runs/maskrcnn_train")


cpu


In [41]:

import cv2
import numpy as np

from utils import rle_to_mask
from torch.utils.data import Dataset
from typing import Dict, List, Optional, Any, Tuple

In [42]:
class ShipDataset(Dataset):
    """
    PyTorch Dataset for instance segmentation of ships.
    Loads images lazily and decodes RLE masks on-demand.
    Returns data in the format expected by Mask R-CNN.
    """

    def __init__(
        self,
        rle_dict: Dict[str, List[str]],
        image_root: str,
        transforms: Optional[Any] = None
    ):
        """
        Initialize the ShipDataset.

        :param rle_dict: Dictionary mapping image filenames to lists of RLE masks.
        :param image_root: Path to the folder containing images.
        :param transforms: Optional torchvision transforms to apply to the image.
        :returns: None
        """
        self.rle_dict = rle_dict
        self.image_root = image_root
        self.transforms = transforms
        self.img_ids = list(rle_dict.keys())

    def __len__(self) -> int:
        """
        Return the number of images in the dataset.

        :returns: Number of images as an integer.
        """
        return len(self.img_ids)

    def __getitem__(self, idx: int) -> Optional[Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Load one image and all its instance masks in Mask R-CNN format.

        :param idx: Index of the image to load.
        :returns: Tuple of (image tensor, target dictionary) or None if no valid masks exist.
        """
        img_id = self.img_ids[idx]
        rles = self.rle_dict[img_id]

        img_path = f"{self.image_root}/{img_id}"
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(img_path)

        img = img[:, :, ::-1]
        img = img.astype(np.float32) / 255.0
        img_tensor = torch.from_numpy(img).permute(2, 0, 1)

        H, W = img_tensor.shape[1:]

        masks = []
        boxes = []

        for rle in rles:
            mask = rle_to_mask(rle, H, W).astype(np.uint8)
            
            if mask.sum() == 0:
                continue
            
            ys, xs = np.where(mask == 1)
            if len(xs) == 0:
                continue
            
            masks.append(mask)

            x1, y1 = xs.min(), ys.min()
            x2, y2 = xs.max(), ys.max()
            boxes.append([x1, y1, x2, y2])
        
        if len(boxes) == 0:
            return None

        masks = torch.as_tensor(np.stack(masks), dtype=torch.uint8)
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }

        return img_tensor, target

In [43]:
from tqdm import tqdm
from torch.utils.data import DataLoader

In [44]:
def train_one_epoch(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    data_loader: DataLoader,
    device: str,
    epoch: int
) -> float:
    """
    Train a Mask R-CNN model for one epoch.

    :param model: Mask R-CNN model.
    :param optimizer: PyTorch optimizer (SGD, Adam, etc.).
    :param data_loader: DataLoader yielding (images, targets).
    :param device: Device to run the training on ('cuda' or 'cpu').
    :param epoch: Current epoch number.
    :returns: Average loss over the epoch as a float.
    """

    model.train()
    total_loss: float = 0.0

    pbar = tqdm(data_loader, desc=f"Epoch {epoch}")

    for batch in pbar:
        
        if batch is None:
            continue

        images, targets = batch
        
        images: list[torch.Tensor] = [img.to(device) for img in images]

        targets: list[dict[str, torch.Tensor]] = [
            {key: val.to(device) for key, val in t.items()}
            for t in targets
        ]

        loss_dict: dict[str, torch.Tensor] = model(images, targets)

        losses: torch.Tensor = sum(loss_dict.values(), torch.tensor(0.0, device=device))
        
        total_loss += losses.item()

        optimizer.zero_grad()   
        losses.backward()
        optimizer.step()

        pbar.set_postfix(loss=float(losses.item()))

    return total_loss / len(data_loader)

In [45]:
import os

from torch.nn import Module

In [46]:
def train(
    model: Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int
) -> None:
    """
    Train a Mask R-CNN model for multiple epochs, log losses, and save checkpoints.

    :param model: Mask R-CNN model to train.
    :param train_loader: DataLoader for training data.
    :param val_loader: DataLoader for validation data (unused in current code).
    :param epochs: Number of epochs to train.
    :returns: None
    """
    os.makedirs("checkpoints", exist_ok=True)
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)

        lr_scheduler.step()

        writer.add_scalar("Loss/train", train_loss, epoch)

        print(f"Epoch {epoch}/{epochs} - Loss: {train_loss:.4f}")

        torch.save(model.state_dict(), f"checkpoints/maskrcnn_epoch_{epoch}.pth")

    writer.close()


In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [48]:
df = pd.read_csv("../../data/segmentations.csv")

rle_dict = (
    df.groupby("ImageId")["EncodedPixels"]
      .apply(list)
      .to_dict()
)

im_ids = list(rle_dict.keys())

im_ids[:3]

test_im_ids, temp_im_ids = train_test_split(im_ids, test_size=0.4, random_state=42)
val_im_ids, train_im_ids = train_test_split(temp_im_ids, test_size=0.5, random_state=42)

train_rle_dict = {im_id: rle_dict[im_id] for im_id in train_im_ids}


shipDataset = ShipDataset(rle_dict=train_rle_dict,
                          image_root="../../data/images",
                          transforms=None)

In [49]:
from numpy.typing import NDArray

In [50]:
def collate_fn(
    batch: List[Tuple[NDArray, dict[str, NDArray]]]
) -> Optional[Tuple[Tuple[NDArray, ...], Tuple[dict[str, NDArray], ...]]]:
    """
    Custom collate function for PyTorch DataLoader that filters out None entries.

    :param batch: List of (image, target) tuples where image is a NumPy array and target is a dictionary of NumPy arrays.
    :returns: Tuple of zipped images and targets, or None if the batch is empty.
    """
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return tuple(zip(*batch))

In [51]:
train(
    model=model,
    train_loader=DataLoader(shipDataset, batch_size=4, shuffle=True, collate_fn=collate_fn),
    val_loader=None,
    epochs=5
)

Epoch 1:   0%|          | 3/9628 [00:11<9:56:45,  3.72s/it, loss=3.38] 


KeyboardInterrupt: 

In [ ]:
from typing import Dict, List
from utils import compute_iou_matrix, average_f_score_of_image, rles_to_masks

In [ ]:
@torch.inference_mode()
def evaluate_model(
    model: torch.nn.Module,
    rle_dict: Dict[str, List[str]],
    image_root: str,
    device: str = "cuda"
) -> Tuple[float, List[float]]:
    """
    Evaluate Faster-RCNN segmentation performance using F2 score.

    :param model: Trained Mask/Faster R-CNN model.
    :param rle_dict: Dictionary mapping image filenames to lists of ground truth RLE masks.
    :param image_root: Path to the folder containing images.
    :param device: Device to run inference on ('cuda' or 'cpu').
    :returns: Tuple of (average F2 score over all images, list of F2 scores per image).
    """
    model.eval()
    f2_scores = []

    for img_id, gt_rles in tqdm(rle_dict.items(), desc="Evaluating"):
        img_path = f"{image_root}/{img_id}"
        img_d = cv2.imread(img_path)
        
        if img_d is None:
            raise FileNotFoundError(img_path)
        
        img = img_d[:, :, ::-1]
        img_tensor = torch.from_numpy(img.astype(np.float32) / 255.).permute(2,0,1).to(device)

        pred = model([img_tensor])[0]  
        pred_masks = pred["masks"].squeeze(1).cpu().numpy() > 0.5

        H, W = img_tensor.shape[1:]
        gt_masks = rles_to_masks(gt_rles, H, W)

        iou_mat = compute_iou_matrix(gt_masks, pred_masks)

        f2 = average_f_score_of_image(iou_mat)
        f2_scores.append(f2)

    return np.mean(f2_scores), f2_scores

In [ ]:
_mean_f2, f2_scores = evaluate_model(
    model=model,
    rle_dict={im_id: rle_dict[im_id] for im_id in val_im_ids},
    image_root="./data/images",
    device="cuda"
)

print(f"Mean F2 Score on Validation Set: {_mean_f2:.4f}")